# 02 — SBERT + BERTopic: Topic Modeling Berita & Ulasan Kesehatan Bangkalan
### Dashboard Identifikasi dan Analisis Tren Isu Kesehatan Masyarakat di Bangkalan

**Tujuan notebook ini:** menemukan struktur/topik laten pada dua corpus yang sudah divalidasi
relevansinya (`berita_modeling.csv`, `ulasan_modeling.csv`), menggunakan SBERT untuk embedding
semantik dan BERTopic (UMAP + HDBSCAN + c-TF-IDF) untuk topic modeling, dibandingkan dengan
baseline TF-IDF + KMeans.

**Notebook ini TIDAK membuat dashboard Streamlit** — itu tahap terpisah setelah notebook ini.

**Pipeline penelitian:**
`data digital → cleaning → validasi relevansi → SBERT embedding → BERTopic → HDBSCAN →
analisis topik → analisis temporal → dashboard Streamlit`. Notebook ini mengerjakan bagian
tengah pipeline (SBERT → HDBSCAN → analisis topik → analisis temporal), memakai output
notebook `01b_relevance_validation.ipynb` sebagai satu-satunya input.

**Prinsip metodologis yang dipegang di seluruh notebook:**
- **BERTopic menemukan struktur/topik laten pada teks — bukan ground truth.** Interpretasi
  topik adalah hasil analisis peneliti (Bagian 11), bukan label otomatis yang final.
- Dataset `raw` dan `cleaned` tidak disentuh sama sekali; notebook ini **hanya membaca**
  dari `data/modeling/`.
- Tidak ada stemming, stopword removal, atau normalisasi agresif — teks natural dipertahankan
  karena SBERT membutuhkan konteks semantik penuh.
- Berita (Corpus A) dan ulasan (Corpus B) dimodelkan **terpisah** — karakteristik keduanya
  berbeda dan tidak diasumsikan topiknya sebanding hanya karena bernomor sama.
- Embedding SBERT dihitung **sekali** dan disimpan ke disk; sel-sel berikutnya memuat ulang
  dari cache alih-alih menghitung ulang (penting untuk efisiensi di Google Colab).
- Semua eksperimen memakai `RANDOM_STATE = 42` agar reproducible, dan konfigurasi akhir
  dicatat ke `model_config.json` untuk ditelusuri kembali saat menulis Bab III/IV skripsi.

**Daftar isi:**
0. Judul, tujuan, konfigurasi (sel ini)
1. Import library & pengecekan versi
2. Load dataset modeling
3. Audit singkat corpus modeling
4. Pemilihan kolom text & metadata
5. Persiapan dokumen
6. Baseline TF-IDF + KMeans
7. SBERT embedding
8. BERTopic (pipeline dasar)
9. HDBSCAN & eksperimen parameterisasi
10. Evaluasi topic model
11. Analisis topik (representasi & interpretasi awal)
12. Analisis temporal
13. Perbandingan berita vs ulasan, dan baseline vs SBERT-BERTopic
14. Simpan seluruh hasil & visualisasi
15. Ringkasan eksperimen, rekomendasi konfigurasi final, dan checklist


In [ ]:
# --- Konfigurasi terpusat (section 18: reproducibility) ---
# Semua angka acak pada notebook ini memakai RANDOM_STATE ini.
RANDOM_STATE = 42

# Kandidat model SBERT -- TIDAK di-hard-code ke satu model saja (lihat Bagian 7).
# Model pertama yang berhasil dimuat yang akan dipakai.
SBERT_MODEL_CANDIDATES = [
    "paraphrase-multilingual-MiniLM-L12-v2",
    "paraphrase-multilingual-mpnet-base-v2",
    "distiluse-base-multilingual-cased-v2",
]

# Kandidat jumlah cluster untuk baseline TF-IDF + KMeans (Bagian 6)
BASELINE_K_CANDIDATES = [5, 7, 10, 12, 15]

# Kandidat min_cluster_size untuk eksperimen HDBSCAN (Bagian 9)
# Dibatasi 4 nilai saja (bukan grid search besar) agar notebook tetap efisien di Colab.
HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES = [10, 15, 20, 30]

# Konfigurasi UMAP dipakai konsisten di semua eksperimen HDBSCAN, supaya HANYA
# min_cluster_size yang menjadi variabel yang diuji (satu variabel per eksperimen,
# bukan grid search UMAP x HDBSCAN penuh -- ASUMSI METODOLOGIS eksplisit).
UMAP_CONFIG = {
    "n_neighbors": 15,
    "n_components": 5,
    "min_dist": 0.0,
    "metric": "cosine",
    "random_state": RANDOM_STATE,
}

print("RANDOM_STATE                 :", RANDOM_STATE)
print("SBERT_MODEL_CANDIDATES       :", SBERT_MODEL_CANDIDATES)
print("BASELINE_K_CANDIDATES        :", BASELINE_K_CANDIDATES)
print("HDBSCAN_MIN_CLUSTER_SIZE_... :", HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES)
print("UMAP_CONFIG                  :", UMAP_CONFIG)


## 1. Import Library & Pengecekan Versi

**Tujuan:** memuat seluruh library yang dipakai notebook ini dan mencatat versinya, agar
eksperimen dapat direproduksi dan dilaporkan di Bab III (lingkungan eksperimen).
**Input:** tidak ada (instalasi/impor). **Output:** namespace siap pakai + tabel versi paket.

Jika sebuah library belum terpasang di runtime Colab Anda, jalankan sel `pip install` di
bawah terlebih dahulu (sekali saja per sesi Colab).


In [ ]:
# Jalankan HANYA jika library belum tersedia di runtime Colab Anda.
# -q untuk output ringkas. Boleh dilewati jika sudah terpasang.
!pip install -q sentence-transformers bertopic hdbscan umap-learn gensim


In [ ]:
import sys
import os
import json as json_lib
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

import sentence_transformers
from sentence_transformers import SentenceTransformer

import bertopic
from bertopic import BERTopic

import hdbscan as hdbscan_lib
from hdbscan import HDBSCAN

import umap as umap_lib
from umap import UMAP

import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

# Coherence topik (Bagian 10) memakai gensim (opsional). Jika tidak tersedia,
# bagian coherence akan dilewati dengan pesan yang jelas, BUKAN disembunyikan.
try:
    import gensim
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False
    print("PERINGATAN: gensim tidak tersedia -- topic coherence (Bagian 10) akan dilewati.")

import importlib.metadata as _ilm

def _versi_paket(nama_modul: str, nama_distribusi: str) -> str:
    """Ambil versi paket dengan aman: coba __version__ dulu, lalu importlib.metadata,
    karena tidak semua paket (mis. hdbscan) mengekspos atribut __version__."""
    modul = sys.modules.get(nama_modul)
    if modul is not None and hasattr(modul, "__version__"):
        return modul.__version__
    try:
        return _ilm.version(nama_distribusi)
    except _ilm.PackageNotFoundError:
        return "(tidak diketahui)"

print("Python version            :", sys.version.split()[0])
print("pandas                     :", pd.__version__)
print("numpy                      :", np.__version__)
print("scikit-learn               :", sklearn.__version__)
print("sentence-transformers      :", _versi_paket("sentence_transformers", "sentence-transformers"))
print("bertopic                   :", _versi_paket("bertopic", "bertopic"))
print("hdbscan                    :", _versi_paket("hdbscan", "hdbscan"))
print("umap-learn                 :", _versi_paket("umap", "umap-learn"))
print("gensim                     :", _versi_paket("gensim", "gensim") if GENSIM_AVAILABLE else "(tidak tersedia)")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device untuk SBERT         :", DEVICE)
if DEVICE == "cpu":
    print("Catatan: GPU tidak terdeteksi. Di Colab, aktifkan Runtime > Change runtime type > GPU "
          "agar SBERT encoding lebih cepat (opsional, tetap berjalan di CPU).")


## 2. Load Dataset Modeling

**Tujuan:** memuat `berita_modeling.csv` dan `ulasan_modeling.csv` dari `01b_relevance_validation.ipynb`
sebagai satu-satunya sumber data. **Input:** `data/modeling/*.csv`. **Output:** `df_berita`,
`df_ulasan` (Corpus A & Corpus B). **Dipakai di:** seluruh bagian berikutnya.

Berita dan ulasan **tidak digabung** pada tahap ini maupun tahap manapun di notebook ini --
keduanya dimodelkan sebagai dua corpus independen (Corpus A = berita, Corpus B = ulasan).


In [ ]:
# Jalankan HANYA di Google Colab. Lewati jika menjalankan secara lokal.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
BASE_DIR = "/content/drive/MyDrive/project"

MODELING_DIR = os.path.join(BASE_DIR, "data", "modeling")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
FIGURES_DIR = os.path.join(PROCESSED_DIR, "figures")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

BERITA_MODELING_PATH = os.path.join(MODELING_DIR, "berita_modeling.csv")
ULASAN_MODELING_PATH = os.path.join(MODELING_DIR, "ulasan_modeling.csv")

# --- Pengecekan file: robust, bukan asumsi diam-diam bahwa file ada (section 22) ---
for path in [BERITA_MODELING_PATH, ULASAN_MODELING_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File tidak ditemukan: {path}\n"
            f"Pastikan notebook 01b_relevance_validation.ipynb sudah dijalankan sampai selesai."
        )

df_berita = pd.read_csv(BERITA_MODELING_PATH, encoding="utf-8-sig")
df_ulasan = pd.read_csv(ULASAN_MODELING_PATH, encoding="utf-8-sig")

# --- Pengecekan kolom minimal (section 22) ---
KOLOM_MINIMAL_BERITA = ["document_id", "source", "text", "tanggal", "tahun", "bulan",
                         "tahun_bulan", "publisher", "kecamatan", "url"]
KOLOM_MINIMAL_ULASAN = ["document_id", "source", "text", "tanggal", "tahun", "bulan",
                         "tahun_bulan", "place_name", "place_category", "rating"]

for nama, df, kolom_wajib in [("berita", df_berita, KOLOM_MINIMAL_BERITA),
                               ("ulasan", df_ulasan, KOLOM_MINIMAL_ULASAN)]:
    kolom_hilang = [k for k in kolom_wajib if k not in df.columns]
    assert not kolom_hilang, f"Kolom wajib hilang pada dataset {nama}: {kolom_hilang}"

# --- Pengecekan document_id unik (section 22: konsistensi document_id) ---
assert df_berita["document_id"].is_unique, "document_id pada df_berita tidak unik!"
assert df_ulasan["document_id"].is_unique, "document_id pada df_ulasan tidak unik!"

print("Corpus A (berita) :", df_berita.shape)
print("Corpus B (ulasan) :", df_ulasan.shape)
print("\nSemua kolom wajib tersedia dan document_id unik pada kedua corpus.")


## 3. Audit Singkat Corpus Modeling

**Tujuan:** memastikan corpus yang akan di-embedding benar-benar bersih (tidak ada text kosong)
dan memahami komposisinya (per tahun, per source, per tahun_bulan) sebelum masuk ke SBERT.
**Input:** `df_berita`, `df_ulasan`. **Output:** ringkasan audit (dicetak) + `assert` yang
menghentikan notebook jika ditemukan text kosong.


In [ ]:
def audit_corpus(df: pd.DataFrame, nama: str) -> None:
    print(f"=== AUDIT CORPUS: {nama} ===")
    print("Jumlah dokumen         :", len(df))

    panjang_karakter = df["text"].fillna("").astype(str).str.len()
    jumlah_kata = df["text"].fillna("").astype(str).str.split().str.len()
    jumlah_kosong = (panjang_karakter == 0).sum()

    print("Jumlah text kosong      :", jumlah_kosong)
    print("Panjang teks (karakter) : min={}, max={}, rata-rata={:.1f}, median={}".format(
        panjang_karakter.min(), panjang_karakter.max(), panjang_karakter.mean(), panjang_karakter.median()
    ))
    print("Jumlah kata             : min={}, max={}, rata-rata={:.1f}, median={}".format(
        jumlah_kata.min(), jumlah_kata.max(), jumlah_kata.mean(), jumlah_kata.median()
    ))

    print("\nJumlah dokumen per tahun:")
    display(df["tahun"].value_counts().sort_index())

    print("\nJumlah dokumen per source:")
    display(df["source"].value_counts())

    print("\nJumlah dokumen per tahun_bulan (10 periode pertama):")
    display(df["tahun_bulan"].value_counts().sort_index().head(10))

    # --- assert text tidak kosong (diminta eksplisit) ---
    assert jumlah_kosong == 0, (
        f"Ditemukan {jumlah_kosong} dokumen dengan text kosong pada corpus {nama}. "
        f"Perbaiki di notebook 01b sebelum melanjutkan -- jangan memfilter di sini."
    )
    print(f"\nOK: tidak ada text kosong pada corpus {nama}.")
    print("=" * 70)


audit_corpus(df_berita, "berita (Corpus A)")
audit_corpus(df_ulasan, "ulasan (Corpus B)")


## 4. Pemilihan Kolom Text & Metadata

**Tujuan:** menegaskan secara eksplisit kolom mana yang menjadi **input embedding** (`text`)
dan kolom mana yang hanya menjadi **metadata** dipakai setelah topic modeling (bukan masuk ke
embedding). Ini mencegah kebocoran metadata ke representasi semantik.
**Input:** `df_berita`, `df_ulasan`. **Output:** daftar `METADATA_COLS_*` yang dipakai ulang di
Bagian 11-13. **Dipakai di:** analisis topik & analisis temporal.


In [ ]:
TEXT_COL = "text"  # satu-satunya kolom yang masuk ke SBERT untuk KEDUA corpus

METADATA_COLS_BERITA = ["document_id", "source", "tanggal", "tahun", "bulan", "tahun_bulan",
                         "publisher", "kecamatan", "url"]
METADATA_COLS_ULASAN = ["document_id", "source", "tanggal", "tahun", "bulan", "tahun_bulan",
                         "place_name", "place_category", "rating"]

print(f"Kolom input embedding (SBERT)     : '{TEXT_COL}'")
print("Kolom metadata berita (non-embedding):", METADATA_COLS_BERITA)
print("Kolom metadata ulasan (non-embedding):", METADATA_COLS_ULASAN)
print("\nMetadata TIDAK digabungkan ke dalam teks yang di-encode SBERT pada notebook ini.")


## 5. Persiapan Dokumen

**Tujuan:** menyiapkan list dokumen (urutan tetap, selaras dengan `document_id`) untuk kedua
corpus, sebagai input langsung ke TF-IDF (Bagian 6) dan SBERT (Bagian 7). **Tidak ada**
stemming, stopword removal, lowercase paksa, atau normalisasi agresif lain di sini -- teks
`text` dari `data/modeling/` sudah merupakan bentuk final yang dipertahankan apa adanya, karena
SBERT membutuhkan konteks kalimat natural.
**Input:** `df_berita`, `df_ulasan`. **Output:** `documents_berita`, `ids_berita`,
`documents_ulasan`, `ids_ulasan`. **Dipakai di:** Bagian 6, 7, 8.


In [ ]:
# Urutan list = urutan baris df saat ini (dipertahankan tetap, TIDAK diacak),
# supaya index embedding <-> document_id selalu bisa direkonstruksi.
documents_berita = df_berita[TEXT_COL].astype(str).tolist()
ids_berita = df_berita["document_id"].tolist()

documents_ulasan = df_ulasan[TEXT_COL].astype(str).tolist()
ids_ulasan = df_ulasan["document_id"].tolist()

assert len(documents_berita) == len(ids_berita) == len(df_berita)
assert len(documents_ulasan) == len(ids_ulasan) == len(df_ulasan)

print(f"documents_berita: {len(documents_berita)} dokumen siap")
print(f"documents_ulasan: {len(documents_ulasan)} dokumen siap")
print("\nContoh dokumen berita:", documents_berita[0][:150])
print("Contoh dokumen ulasan:", documents_ulasan[0][:150])


## 6. Baseline TF-IDF + KMeans

**Tujuan:** membangun baseline metodologis sederhana (representasi bag-of-words + clustering
klasik) sebagai pembanding SBERT + BERTopic (Bagian 13). Baseline TIDAK memakai embedding
semantik -- hanya frekuensi kata (TF-IDF), sehingga tidak menangkap sinonim/parafrase.
**Input:** `documents_berita`, `documents_ulasan`. **Output:** `baseline_results.csv` (silhouette,
Calinski-Harabasz, Davies-Bouldin untuk tiap kandidat k). **Dipakai di:** Bagian 13
(perbandingan baseline vs SBERT-BERTopic).

Beberapa kandidat k diuji (bukan hanya satu) karena jumlah cluster optimal bergantung pada
metrik yang dipakai -- keputusan k TIDAK diambil dari satu metrik saja.


In [ ]:
def jalankan_baseline_tfidf_kmeans(documents: list, k_list: list, dataset_name: str,
                                    random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """
    Membangun representasi TF-IDF lalu clustering KMeans untuk beberapa nilai k.
    Mengembalikan DataFrame hasil evaluasi (silhouette, calinski_harabasz, davies_bouldin)
    untuk tiap k -- TIDAK memilih k "terbaik" secara otomatis (keputusan ada di Bagian 13/17).
    """
    vectorizer = TfidfVectorizer(max_features=5000, min_df=2)
    X_tfidf = vectorizer.fit_transform(documents)
    print(f"[{dataset_name}] Matriks TF-IDF: {X_tfidf.shape[0]} dokumen x {X_tfidf.shape[1]} fitur")

    hasil = []
    for k in k_list:
        t0 = time.time()
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        label = km.fit_predict(X_tfidf)
        durasi = time.time() - t0

        # Silhouette pada data sparse berdimensi tinggi bisa lambat -- gunakan sample_size
        # untuk menjaga efisiensi Colab (section 19), tetap reproducible via random_state.
        n_sample = min(2000, X_tfidf.shape[0])
        sil = silhouette_score(X_tfidf, label, sample_size=n_sample, random_state=random_state)
        ch = calinski_harabasz_score(X_tfidf.toarray(), label)
        db = davies_bouldin_score(X_tfidf.toarray(), label)

        hasil.append({
            "dataset": dataset_name, "k": k,
            "silhouette": round(sil, 4), "calinski_harabasz": round(ch, 2),
            "davies_bouldin": round(db, 4), "waktu_proses_detik": round(durasi, 2),
        })
        print(f"  k={k:>2}: silhouette={sil:.4f}, calinski_harabasz={ch:.2f}, "
              f"davies_bouldin={db:.4f} ({durasi:.2f}s)")

    return pd.DataFrame(hasil)


t0_baseline = time.time()
baseline_berita = jalankan_baseline_tfidf_kmeans(documents_berita, BASELINE_K_CANDIDATES, "berita")
baseline_ulasan = jalankan_baseline_tfidf_kmeans(documents_ulasan, BASELINE_K_CANDIDATES, "ulasan")
baseline_results = pd.concat([baseline_berita, baseline_ulasan], ignore_index=True)

print(f"\nTotal waktu baseline: {time.time() - t0_baseline:.1f} detik")
display(baseline_results)


**Interpretasi metodologis (bukan kesimpulan "terbaik"):**
- **Silhouette** mengukur seberapa rapat sebuah dokumen ke cluster-nya dibanding cluster
  terdekat lain (rentang -1 s.d. 1, lebih tinggi umumnya lebih baik).
- **Calinski-Harabasz** membandingkan sebaran antar-cluster terhadap sebaran dalam-cluster
  (lebih tinggi umumnya lebih baik), tetapi cenderung menyukai jumlah cluster yang lebih besar
  pada beberapa jenis data.
- **Davies-Bouldin** mengukur rata-rata kemiripan tiap cluster dengan cluster paling mirip
  (lebih rendah lebih baik).
- Karena ketiga metrik bisa tidak sepakat satu sama lain (mis. k tertentu terbaik pada
  silhouette tapi tidak pada davies_bouldin), k baseline dipilih di Bagian 13/17 dengan
  mempertimbangkan ketiganya bersama, bukan satu metrik saja.
- Baseline ini memakai representasi TF-IDF (bag-of-words) sehingga secara metodologis TIDAK
  bisa menangkap kemiripan semantik/parafrase antar dokumen -- ini justru menjadi salah satu
  pembeda utama yang dibandingkan dengan SBERT + BERTopic pada Bagian 13.


In [ ]:
BASELINE_RESULTS_PATH = os.path.join(PROCESSED_DIR, "baseline_results.csv")
baseline_results.to_csv(BASELINE_RESULTS_PATH, index=False, encoding="utf-8-sig")
print("Tersimpan:", BASELINE_RESULTS_PATH)


## 7. SBERT Embedding

**Tujuan:** mengubah setiap dokumen menjadi vektor semantik (embedding) memakai Sentence-BERT,
model transformer yang cocok untuk bahasa Indonesia/multilingual. **Input:** `documents_berita`,
`documents_ulasan`. **Output:** `embeddings_berita.npy`, `embeddings_ulasan.npy` +
`embedding_index_berita.csv`, `embedding_index_ulasan.csv` (pemetaan baris embedding <->
`document_id`, agar embedding bisa direkonstruksi dengan benar kapan pun). **Dipakai di:**
Bagian 8 dan seterusnya (BERTopic memakai embedding ini, TIDAK menghitung ulang).

Model dicoba satu per satu dari `SBERT_MODEL_CANDIDATES` (bukan hard-code satu model) --
model pertama yang berhasil dimuat yang dipakai untuk KEDUA corpus (supaya embedding
berita & ulasan berada di ruang vektor yang sama, memudahkan perbandingan metodologis).

**Jika file embedding sudah ada, sel ini TIDAK menghitung ulang** -- langsung memuat dari
cache. Hapus file `.npy` terkait secara sengaja jika ingin menghitung ulang dari awal.


In [ ]:
def muat_model_sbert(kandidat: list) -> tuple:
    """
    Mencoba memuat model SBERT dari daftar kandidat secara berurutan.
    Mengembalikan (model, nama_model_yang_berhasil).
    Berhenti pada kandidat pertama yang berhasil dimuat.
    """
    for nama_model in kandidat:
        try:
            print(f"Mencoba memuat model SBERT: {nama_model} ...")
            model = SentenceTransformer(nama_model, device=DEVICE)
            print(f"  Berhasil dimuat: {nama_model}")
            return model, nama_model
        except Exception as e:
            print(f"  Gagal memuat {nama_model}: {e}")
    raise RuntimeError(
        "Tidak ada model SBERT dari SBERT_MODEL_CANDIDATES yang berhasil dimuat. "
        "Periksa koneksi internet Colab atau perbarui daftar kandidat."
    )


sbert_model, SBERT_MODEL_NAME = muat_model_sbert(SBERT_MODEL_CANDIDATES)
EMBEDDING_DIM = sbert_model.get_sentence_embedding_dimension()
print(f"\nModel SBERT dipakai : {SBERT_MODEL_NAME}")
print(f"Dimensi embedding    : {EMBEDDING_DIM}")


In [ ]:
def hitung_atau_muat_embedding(documents: list, ids: list, nama_dataset: str,
                                model: SentenceTransformer, embedding_path: str,
                                index_path: str) -> np.ndarray:
    """
    Jika embedding_path sudah ada, muat langsung dari cache (TIDAK menghitung ulang).
    Jika belum ada, hitung dengan SBERT, simpan .npy + index csv (document_id per baris).
    """
    if os.path.exists(embedding_path) and os.path.exists(index_path):
        print(f"[{nama_dataset}] Cache ditemukan, memuat dari: {embedding_path}")
        embeddings = np.load(embedding_path)
        index_df = pd.read_csv(index_path)

        # Pengecekan konsistensi: index cache harus sama persis dengan document_id saat ini
        assert list(index_df["document_id"]) == list(ids), (
            f"document_id pada cache {index_path} tidak sama dengan urutan dokumen saat ini. "
            f"Hapus file cache ini jika dataset modeling sudah berubah, lalu jalankan ulang sel ini."
        )
        assert embeddings.shape[0] == len(ids), "Jumlah embedding != jumlah dokumen pada cache!"
        return embeddings

    print(f"[{nama_dataset}] Cache tidak ditemukan, menghitung embedding untuk "
          f"{len(documents)} dokumen ...")
    t0 = time.time()
    embeddings = model.encode(documents, show_progress_bar=True, batch_size=64,
                               convert_to_numpy=True)
    durasi = time.time() - t0
    print(f"[{nama_dataset}] Selesai dalam {durasi:.1f} detik "
          f"({durasi / max(len(documents), 1):.3f} detik/dokumen, device={DEVICE})")

    assert embeddings.shape[0] == len(documents), "Jumlah embedding != jumlah dokumen!"

    np.save(embedding_path, embeddings)
    pd.DataFrame({"row_index": range(len(ids)), "document_id": ids}).to_csv(index_path, index=False)
    print(f"[{nama_dataset}] Tersimpan: {embedding_path}, {index_path}")
    return embeddings


EMBEDDINGS_BERITA_PATH = os.path.join(PROCESSED_DIR, "embeddings_berita.npy")
EMBEDDINGS_ULASAN_PATH = os.path.join(PROCESSED_DIR, "embeddings_ulasan.npy")
EMBEDDING_INDEX_BERITA_PATH = os.path.join(PROCESSED_DIR, "embedding_index_berita.csv")
EMBEDDING_INDEX_ULASAN_PATH = os.path.join(PROCESSED_DIR, "embedding_index_ulasan.csv")

embeddings_berita = hitung_atau_muat_embedding(
    documents_berita, ids_berita, "berita", sbert_model,
    EMBEDDINGS_BERITA_PATH, EMBEDDING_INDEX_BERITA_PATH,
)
embeddings_ulasan = hitung_atau_muat_embedding(
    documents_ulasan, ids_ulasan, "ulasan", sbert_model,
    EMBEDDINGS_ULASAN_PATH, EMBEDDING_INDEX_ULASAN_PATH,
)

print("\nembeddings_berita:", embeddings_berita.shape)
print("embeddings_ulasan:", embeddings_ulasan.shape)


### 7.1 Normalisasi Embedding

BERTopic secara default memakai **cosine similarity** (melalui UMAP dengan `metric="cosine"`,
lihat `UMAP_CONFIG`) untuk mengukur kemiripan antar embedding. Cosine similarity **tidak
mensyaratkan** vektor dinormalisasi ke panjang 1 (ia sudah membagi dengan norma masing-masing
vektor secara internal), sehingga **normalisasi eksplisit tidak dilakukan** di sini --
normalisasi tambahan berisiko mengubah skala relatif antar dimensi tanpa manfaat tambahan
untuk metric cosine. Jika di kemudian hari dipakai metric Euclidean (bukan cosine), embedding
SBERT sebaiknya dinormalisasi (L2-normalize) terlebih dahulu -- keputusan ini didokumentasikan
di sini agar konsisten dan bisa ditelusuri.


In [ ]:
print("=== Dokumentasi lingkungan embedding (untuk Bab III) ===")
print("Model SBERT       :", SBERT_MODEL_NAME)
print("Dimensi embedding :", EMBEDDING_DIM)
print("Jumlah dokumen    : berita =", len(ids_berita), ", ulasan =", len(ids_ulasan))
print("Device            :", DEVICE)
print("Metric similarity :", "cosine (dipakai di UMAP, embedding TIDAK dinormalisasi eksplisit)")


## 8. BERTopic (Pipeline Dasar)

**Tujuan:** menyusun fungsi pembangun model BERTopic yang memakai embedding SBERT yang SUDAH
dihitung (Bagian 7) -- BERTopic tidak dibiarkan menghitung ulang embedding. **Input:**
`documents_*`, `embeddings_*`. **Output:** fungsi `bangun_topic_model(...)` yang dipakai
berulang pada Bagian 9 (eksperimen HDBSCAN). **Dipakai di:** Bagian 9-14.

Karena embedding sudah dihitung dan diberikan langsung ke `fit_transform(docs, embeddings=...)`,
`embedding_model` di-set `None` -- BERTopic tidak akan memuat/menjalankan model SBERT apa pun
secara internal.


In [ ]:
def bangun_topic_model(documents: list, embeddings: np.ndarray, min_cluster_size: int,
                        umap_config: dict = UMAP_CONFIG, random_state: int = RANDOM_STATE):
    """
    Membangun & fit satu model BERTopic dengan embedding SBERT yang sudah dihitung
    (embedding_model=None -> BERTopic TIDAK menghitung ulang embedding).

    UMAP dipakai dengan konfigurasi yang SAMA di semua eksperimen (lihat UMAP_CONFIG) --
    hanya min_cluster_size HDBSCAN yang bervariasi antar eksperimen.

    Return: (topic_model, topics, probs, durasi_detik)
    """
    assert embeddings.shape[0] == len(documents), "Jumlah embedding != jumlah dokumen!"

    umap_model = UMAP(
        n_neighbors=umap_config["n_neighbors"],
        n_components=umap_config["n_components"],
        min_dist=umap_config["min_dist"],
        metric=umap_config["metric"],
        random_state=umap_config["random_state"],
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=None,               # default: samakan dengan min_cluster_size
        metric="euclidean",             # dihitung pada ruang hasil reduksi UMAP
        cluster_selection_method="eom",
        prediction_data=True,
    )

    topic_model = BERTopic(
        embedding_model=None,           # embedding SBERT sudah dihitung -- jangan hitung ulang
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        calculate_probabilities=True,
        verbose=False,
    )

    t0 = time.time()
    topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)
    durasi = time.time() - t0

    return topic_model, topics, probs, durasi


print("Fungsi bangun_topic_model() siap dipakai pada Bagian 9.")
print("Konfigurasi UMAP yang dipakai konsisten di semua eksperimen:", UMAP_CONFIG)


## 9-10. HDBSCAN: Eksperimen Parameterisasi & Evaluasi Topic Model

**Tujuan:** menjalankan BERTopic dengan beberapa nilai `min_cluster_size` HDBSCAN
(`HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES = [10, 15, 20, 30]`) untuk KEDUA corpus, lalu
mengevaluasi tiap hasil dengan lebih dari sekadar jumlah topik. **Input:** `documents_*`,
`embeddings_*`. **Output:** `topic_model_results.csv` (satu file gabungan yang memuat kolom
eksperimen HDBSCAN -- jumlah topik/outlier -- SEKALIGUS kolom evaluasi kualitas topik --
diversity/coherence -- karena keduanya dihitung dari model yang sama, sehingga digabung jadi
satu tabel alih-alih dua tabel terpisah yang tumpang tindih). **Dipakai di:** Bagian 11-14
(pemilihan model final memakai tabel ini).

Hanya 4 konfigurasi per corpus (bukan grid search besar) -- sesuai batasan efisiensi Colab.
`topic -1` (outlier HDBSCAN) dipisahkan secara eksplisit dari topik reguler di setiap
perhitungan metrik (topik -1 TIDAK diperlakukan sebagai topik kesehatan/topik biasa).


In [ ]:
def hitung_topic_diversity(topic_model: BERTopic, top_n: int = 10) -> float:
    """
    Topic diversity = proporsi kata UNIK di antara top-N kata semua topik REGULER
    (topik -1/outlier dikecualikan). Rentang 0-1, lebih tinggi berarti topik lebih
    berbeda satu sama lain (kata top tidak banyak tumpang tindih antar topik).
    """
    semua_topik = topic_model.get_topics()
    topik_reguler = {tid: kata for tid, kata in semua_topik.items() if tid != -1}
    if not topik_reguler:
        return float("nan")

    semua_kata_top = []
    for tid, daftar_kata_skor in topik_reguler.items():
        kata_top_n = [kata for kata, skor in daftar_kata_skor[:top_n]]
        semua_kata_top.extend(kata_top_n)

    if not semua_kata_top:
        return float("nan")
    return len(set(semua_kata_top)) / len(semua_kata_top)


def hitung_topic_coherence_umass(topic_model: BERTopic, documents: list, top_n: int = 10):
    """
    Coherence u_mass: HANYA memakai statistik co-occurrence di dalam corpus ini sendiri
    (tidak butuh korpus referensi eksternal), sehingga cepat dihitung tapi juga TERBATAS --
    nilainya relatif (dipakai untuk membandingkan model PADA CORPUS YANG SAMA), biasanya
    negatif, dan semakin dekat ke 0 semakin baik. TIDAK sebanding lintas corpus/dataset
    berbeda. Tokenisasi di sini HANYA untuk keperluan komputasi coherence -- tidak mengubah
    teks yang dipakai bagian manapun yang lain.
    Return: nilai float, atau None jika gensim tidak tersedia / topik reguler kosong.
    """
    if not GENSIM_AVAILABLE:
        return None

    semua_topik = topic_model.get_topics()
    topik_reguler = {tid: kata for tid, kata in semua_topik.items() if tid != -1}
    if not topik_reguler:
        return None

    daftar_top_words = [[kata for kata, skor in daftar[:top_n]] for daftar in topik_reguler.values()]

    tokenized_docs = [d.lower().split() for d in documents]
    dictionary = Dictionary(tokenized_docs)
    corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_docs]

    cm = CoherenceModel(topics=daftar_top_words, corpus=corpus_bow, dictionary=dictionary,
                         coherence="u_mass")
    return cm.get_coherence()


def ringkas_topic_size_distribution(topic_info: pd.DataFrame) -> str:
    """Ringkasan ukuran topik REGULER (tanpa -1) sebagai string ringkas untuk 1 sel CSV."""
    ukuran = topic_info.loc[topic_info["Topic"] != -1, "Count"]
    if len(ukuran) == 0:
        return "tidak ada topik reguler"
    return (f"min={ukuran.min()}, median={ukuran.median():.0f}, "
            f"max={ukuran.max()}, n_topik={len(ukuran)}")


def jalankan_eksperimen_hdbscan(documents: list, embeddings: np.ndarray, dataset_name: str,
                                 min_cluster_size_list: list) -> tuple:
    """
    Menjalankan BERTopic untuk setiap min_cluster_size, mengumpulkan metrik evaluasi.
    Return: (dataframe_hasil, dict_model_per_min_cluster_size, dict_topics_per_min_cluster_size)
    """
    hasil = []
    model_per_config = {}
    topics_per_config = {}

    for mcs in min_cluster_size_list:
        print(f"[{dataset_name}] Menjalankan BERTopic dengan min_cluster_size={mcs} ...")
        topic_model, topics, probs, durasi = bangun_topic_model(documents, embeddings, mcs)

        topic_info = topic_model.get_topic_info()
        jumlah_topic = int((topic_info["Topic"] != -1).sum())
        jumlah_outlier = int(sum(1 for t in topics if t == -1))
        proporsi_outlier = jumlah_outlier / len(topics) if len(topics) > 0 else float("nan")

        diversity = hitung_topic_diversity(topic_model)
        coherence = hitung_topic_coherence_umass(topic_model, documents)

        model_id = f"{dataset_name}_mcs{mcs}"
        hasil.append({
            "dataset": dataset_name,
            "model_id": model_id,
            "min_topic_size": mcs,   # nama kolom mengikuti istilah BERTopic (min_topic_size ~ min_cluster_size HDBSCAN)
            "jumlah_topic": jumlah_topic,
            "jumlah_outlier": jumlah_outlier,
            "proporsi_outlier": round(proporsi_outlier, 4),
            "topic_size_distribution": ringkas_topic_size_distribution(topic_info),
            "topic_diversity": round(diversity, 4) if diversity == diversity else None,  # NaN check
            "coherence": round(coherence, 4) if coherence is not None else None,
            "waktu_proses_detik": round(durasi, 2),
        })
        print(f"  -> {jumlah_topic} topik, {jumlah_outlier} outlier "
              f"({proporsi_outlier:.1%}), diversity={diversity:.3f}, "
              f"coherence={coherence if coherence is not None else 'n/a'} ({durasi:.1f}s)")

        model_per_config[mcs] = topic_model
        topics_per_config[mcs] = (topics, probs)

    return pd.DataFrame(hasil), model_per_config, topics_per_config


t0_hdbscan = time.time()
hasil_berita, model_berita_per_mcs, topics_berita_per_mcs = jalankan_eksperimen_hdbscan(
    documents_berita, embeddings_berita, "berita", HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES
)
hasil_ulasan, model_ulasan_per_mcs, topics_ulasan_per_mcs = jalankan_eksperimen_hdbscan(
    documents_ulasan, embeddings_ulasan, "ulasan", HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES
)
topic_model_results = pd.concat([hasil_berita, hasil_ulasan], ignore_index=True)

print(f"\nTotal waktu eksperimen HDBSCAN: {time.time() - t0_hdbscan:.1f} detik")
display(topic_model_results)


In [ ]:
TOPIC_MODEL_RESULTS_PATH = os.path.join(PROCESSED_DIR, "topic_model_results.csv")
topic_model_results.to_csv(TOPIC_MODEL_RESULTS_PATH, index=False, encoding="utf-8-sig")
print("Tersimpan:", TOPIC_MODEL_RESULTS_PATH)


**Catatan evaluasi (interpretabilitas & keterbatasan coherence):**
- **Topic diversity** tinggi berarti topik-topik cenderung memakai kumpulan kata top yang
  berbeda-beda (bagus untuk keterpisahan topik), TAPI tidak menjamin topik tersebut bermakna
  secara substantif -- itu sebabnya interpretabilitas tetap diperiksa manual di Bagian 11 lewat
  representative documents, bukan hanya top words.
- **Coherence (u_mass)** di sini bersifat internal-corpus: dua model pada corpus BERBEDA
  (mis. berita vs ulasan) tidak boleh dibandingkan langsung nilai coherence-nya -- hanya
  dipakai untuk membandingkan beberapa konfigurasi PADA corpus yang SAMA.
- **Interpretabilitas** topik pada akhirnya diperiksa manusia lewat top words + representative
  documents (Bagian 11) -- metrik numerik di atas adalah alat bantu, bukan pengganti pembacaan
  peneliti.
- **Proporsi outlier** yang tinggi tidak otomatis berarti model buruk (dokumen memang bisa tidak
  masuk topik manapun secara wajar), tetapi proporsi yang SANGAT tinggi (>50%, misalnya) patut
  dicurigai sebagai sinyal `min_cluster_size` terlalu besar untuk ukuran corpus.


## 11. Analisis Topik: Representasi & Interpretasi Awal

**Tujuan:** untuk model yang dipilih (lihat Bagian 13/17 untuk mekanisme pemilihan; sel di
bawah memakai `min_cluster_size` PALING BESAR pada `HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES` sebagai
titik awal analisis mendalam -- **bukan klaim ini konfigurasi terbaik**, hanya representatif
untuk didemonstrasikan; ganti `MCS_TERPILIH_*` di bawah sesuai hasil Bagian 17 sebelum menulis
laporan akhir) menampilkan topic number, topic size, top words, dan representative documents.
**Input:** model dari Bagian 9. **Output:** `topic_summary_berita.csv`, `topic_summary_ulasan.csv`
(kolom `interpretasi_awal` dikosongkan untuk diisi peneliti). **Dipakai di:** Bagian 12 (temporal),
Bagian 14 (penulisan Bab IV).

**Topik -1 (outlier) dilaporkan terpisah** dan tidak dianggap sebagai topik kesehatan atau
topik substantif apa pun.


In [ ]:
# GANTI nilai ini sesuai hasil pemilihan model final di Bagian 17 sebelum menulis laporan akhir.
MCS_TERPILIH_BERITA = max(HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES)
MCS_TERPILIH_ULASAN = max(HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES)

topic_model_berita = model_berita_per_mcs[MCS_TERPILIH_BERITA]
topics_berita, probs_berita = topics_berita_per_mcs[MCS_TERPILIH_BERITA]

topic_model_ulasan = model_ulasan_per_mcs[MCS_TERPILIH_ULASAN]
topics_ulasan, probs_ulasan = topics_ulasan_per_mcs[MCS_TERPILIH_ULASAN]

print(f"Analisis topik memakai min_cluster_size={MCS_TERPILIH_BERITA} (berita), "
      f"min_cluster_size={MCS_TERPILIH_ULASAN} (ulasan).")


In [ ]:
def buat_topic_summary(topic_model: BERTopic, topics: list, documents: list, ids: list,
                        top_n_kata: int = 10, top_n_dokumen: int = 3,
                        panjang_preview: int = 120) -> pd.DataFrame:
    """
    Menyusun tabel ringkasan topik: topic, topic_size, top_words, representative_document_ids,
    representative_text_preview (potongan singkat, BUKAN teks penuh), dan interpretasi_awal
    (dikosongkan -- diisi peneliti, BUKAN label otomatis).
    Topik -1 (outlier) ikut ditampilkan sebagai baris terpisah, bukan dianggap topik biasa.
    """
    topic_info = topic_model.get_topic_info()
    doc_id_by_pos = {i: doc_id for i, doc_id in enumerate(ids)}
    text_by_pos = {i: documents[i] for i in range(len(documents))}

    baris = []
    for _, row in topic_info.iterrows():
        tid = row["Topic"]
        topic_size = row["Count"]

        if tid == -1:
            top_words = "(outlier -- tidak ada representasi kata topik)"
        else:
            top_words = ", ".join([kata for kata, skor in topic_model.get_topic(tid)[:top_n_kata]])

        posisi_dokumen_topik = [i for i, t in enumerate(topics) if t == tid][:top_n_dokumen]
        rep_ids = [doc_id_by_pos[i] for i in posisi_dokumen_topik]
        rep_preview = [text_by_pos[i][:panjang_preview].strip() + "..." for i in posisi_dokumen_topik]

        baris.append({
            "topic": tid,
            "topic_size": topic_size,
            "top_words": top_words,
            "representative_document_ids": "; ".join(rep_ids),
            "representative_text_preview": " || ".join(rep_preview),
            "interpretasi_awal": "",  # dikosongkan -- diisi peneliti setelah membaca representative docs
        })

    return pd.DataFrame(baris)


topic_summary_berita = buat_topic_summary(topic_model_berita, topics_berita, documents_berita, ids_berita)
topic_summary_ulasan = buat_topic_summary(topic_model_ulasan, topics_ulasan, documents_ulasan, ids_ulasan)

print("=== Ringkasan topik berita ===")
display(topic_summary_berita.head(10))
print("\n=== Ringkasan topik ulasan ===")
display(topic_summary_ulasan.head(10))


In [ ]:
TOPIC_SUMMARY_BERITA_PATH = os.path.join(PROCESSED_DIR, "topic_summary_berita.csv")
TOPIC_SUMMARY_ULASAN_PATH = os.path.join(PROCESSED_DIR, "topic_summary_ulasan.csv")

topic_summary_berita.to_csv(TOPIC_SUMMARY_BERITA_PATH, index=False, encoding="utf-8-sig")
topic_summary_ulasan.to_csv(TOPIC_SUMMARY_ULASAN_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", TOPIC_SUMMARY_BERITA_PATH)
print("Tersimpan:", TOPIC_SUMMARY_ULASAN_PATH)
print("\nCara mengisi kolom 'interpretasi_awal': baca top_words + representative_text_preview "
      "(atau buka dokumen lengkap lewat representative_document_ids pada data/modeling/), lalu "
      "tulis 1-2 kalimat tentang isu apa yang tampaknya dibahas topik tersebut. Ini adalah "
      "interpretasi peneliti, BUKAN label ground truth otomatis.")


## 12. Analisis Temporal

**Tujuan:** menggabungkan hasil assignment topik dengan metadata asli (tahun, bulan,
tahun_bulan) untuk melihat bagaimana proporsi tiap topik berubah dari waktu ke waktu.
**Input:** `topics_berita`, `topics_ulasan`, `df_berita`, `df_ulasan`. **Output:**
`document_topic_berita.csv`, `document_topic_ulasan.csv`, `topic_temporal_berita.csv`,
`topic_temporal_ulasan.csv`. **Dipakai di:** Bagian 14 (visualisasi topic-over-time), Bab IV.

Proporsi (bukan hanya jumlah absolut) dihitung karena jumlah dokumen antarperiode bisa
berbeda jauh (mis. tahun dengan liputan media lebih banyak).


In [ ]:
def buat_document_topic(df: pd.DataFrame, ids: list, topics: list, probs) -> pd.DataFrame:
    """
    Menggabungkan document_id + topic assignment + probabilitas topik (jika tersedia)
    dengan metadata waktu (tahun, bulan, tahun_bulan) dari df asli.
    """
    # BERTopic dengan calculate_probabilities=True mengembalikan matriks probabilitas
    # per topik reguler; probabilitas topik yang DIPILIH diambil sebagai topic_probability.
    # Untuk dokumen outlier (-1), probabilitas maksimum tetap dilaporkan apa adanya.
    if probs is not None and hasattr(probs, "ndim") and probs.ndim == 2 and probs.shape[1] > 0:
        topic_probability = probs.max(axis=1)
    else:
        # None, atau array 2D tanpa kolom topik reguler sama sekali (semua dokumen outlier)
        topic_probability = [None] * len(ids)

    df_topic = pd.DataFrame({
        "document_id": ids,
        "topic": topics,
        "topic_probability": topic_probability,
    })
    df_topic = df_topic.merge(df[["document_id", "tahun", "bulan", "tahun_bulan"]],
                               on="document_id", how="left")
    return df_topic


document_topic_berita = buat_document_topic(df_berita, ids_berita, topics_berita, probs_berita)
document_topic_ulasan = buat_document_topic(df_ulasan, ids_ulasan, topics_ulasan, probs_ulasan)

assert len(document_topic_berita) == len(df_berita)
assert len(document_topic_ulasan) == len(df_ulasan)

DOCUMENT_TOPIC_BERITA_PATH = os.path.join(PROCESSED_DIR, "document_topic_berita.csv")
DOCUMENT_TOPIC_ULASAN_PATH = os.path.join(PROCESSED_DIR, "document_topic_ulasan.csv")
document_topic_berita.to_csv(DOCUMENT_TOPIC_BERITA_PATH, index=False, encoding="utf-8-sig")
document_topic_ulasan.to_csv(DOCUMENT_TOPIC_ULASAN_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", DOCUMENT_TOPIC_BERITA_PATH)
print("Tersimpan:", DOCUMENT_TOPIC_ULASAN_PATH)
display(document_topic_berita.head(5))


In [ ]:
def buat_topic_temporal(df_topic: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """
    Menghitung jumlah dokumen dan proporsi tiap topik per periode, pada tiga granularitas
    (tahun, bulan, tahun_bulan), digabung jadi satu tabel long-format dengan kolom
    'granularitas' agar tetap satu file per corpus (bukan 3 file x 2 corpus = 6 file).
    """
    hasil = []
    for granularitas, kolom_periode in [("tahun", "tahun"), ("bulan", "bulan"), ("tahun_bulan", "tahun_bulan")]:
        agg = (
            df_topic.groupby([kolom_periode, "topic"]).size()
            .rename("jumlah_dokumen").reset_index()
        )
        total_per_periode = df_topic.groupby(kolom_periode).size().rename("total_dokumen_periode")
        agg = agg.merge(total_per_periode, on=kolom_periode)
        agg["proporsi_topic_dalam_periode"] = (agg["jumlah_dokumen"] / agg["total_dokumen_periode"]).round(4)
        agg.insert(0, "granularitas", granularitas)
        agg = agg.rename(columns={kolom_periode: "periode"})
        hasil.append(agg[["granularitas", "periode", "topic", "jumlah_dokumen",
                           "total_dokumen_periode", "proporsi_topic_dalam_periode"]])

    hasil_gabungan = pd.concat(hasil, ignore_index=True)
    print(f"[{dataset_name}] topic_temporal: {len(hasil_gabungan)} baris "
          f"(3 granularitas x kombinasi periode-topik yang muncul)")
    return hasil_gabungan


topic_temporal_berita = buat_topic_temporal(document_topic_berita, "berita")
topic_temporal_ulasan = buat_topic_temporal(document_topic_ulasan, "ulasan")

TOPIC_TEMPORAL_BERITA_PATH = os.path.join(PROCESSED_DIR, "topic_temporal_berita.csv")
TOPIC_TEMPORAL_ULASAN_PATH = os.path.join(PROCESSED_DIR, "topic_temporal_ulasan.csv")
topic_temporal_berita.to_csv(TOPIC_TEMPORAL_BERITA_PATH, index=False, encoding="utf-8-sig")
topic_temporal_ulasan.to_csv(TOPIC_TEMPORAL_ULASAN_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", TOPIC_TEMPORAL_BERITA_PATH)
print("Tersimpan:", TOPIC_TEMPORAL_ULASAN_PATH)
display(topic_temporal_berita[topic_temporal_berita["granularitas"] == "tahun"].head(10))


## 13. Perbandingan Model & Pemilihan Konfigurasi Final

**Tujuan:** (a) membandingkan topik berita vs ulasan secara deskriptif (bukan asumsi nomor
topik yang sama = topik yang sama), (b) membandingkan baseline TF-IDF+KMeans vs SBERT+BERTopic
secara konseptual & empiris, (c) memilih konfigurasi `min_cluster_size` final per corpus dengan
rubrik yang bisa ditelusuri (bukan hanya karena jumlah topik "terlihat bagus").
**Input:** `topic_model_results`, `baseline_results`, `topic_summary_*`.
**Output:** `final_model_selection.csv`. **Dipakai di:** Bagian 14-15, Bab IV.


### 13.1 Berita vs Ulasan (Deskriptif)

Topik pada model berita dan model ulasan berasal dari DUA model BERTopic yang terpisah,
dengan ruang UMAP dan cluster HDBSCAN masing-masing sendiri. **Topic 1 pada berita TIDAK
otomatis sama dengan Topic 1 pada ulasan** -- perbandingan yang valid dilakukan lewat
representasi kata (top_words) dan isi (representative documents), bukan nomor topik.


In [ ]:
print("Jumlah topik reguler -- berita:", (topic_summary_berita['topic'] != -1).sum(),
      "| ulasan:", (topic_summary_ulasan['topic'] != -1).sum())

print("\nContoh perbandingan top_words topik terbesar tiap corpus (bukan klaim topik yang sama):")
top_berita = topic_summary_berita[topic_summary_berita['topic'] != -1].nlargest(3, 'topic_size')
top_ulasan = topic_summary_ulasan[topic_summary_ulasan['topic'] != -1].nlargest(3, 'topic_size')
print("\n[berita] 3 topik terbesar:")
display(top_berita[['topic', 'topic_size', 'top_words']])
print("\n[ulasan] 3 topik terbesar:")
display(top_ulasan[['topic', 'topic_size', 'top_words']])
print("\nBandingkan top_words di atas secara kualitatif (mis. apakah ada tema kebersihan "
      "lingkungan yang muncul di kedua corpus, meski dengan kata top yang berbeda) -- "
      "kesimpulan kesamaan/perbedaan tema ditulis peneliti di Bagian 11/15, bukan otomatis di sini.")


### 13.2 Baseline (TF-IDF + KMeans) vs SBERT + BERTopic


In [ ]:
perbandingan_metode = pd.DataFrame([
    {"aspek": "Representasi teks",
     "TF-IDF + KMeans": "Frekuensi kata (bag-of-words), tidak menangkap sinonim/parafrase",
     "SBERT + BERTopic": "Embedding semantik kontekstual, menangkap kemiripan makna meski kata berbeda"},
    {"aspek": "Metode clustering",
     "TF-IDF + KMeans": "KMeans (partisi, jumlah cluster k harus ditentukan di awal)",
     "SBERT + BERTopic": "HDBSCAN (density-based, jumlah topik ditemukan otomatis dari data)"},
    {"aspek": "Menangkap semantic similarity",
     "TF-IDF + KMeans": "Terbatas -- hanya berdasarkan kata yang sama persis",
     "SBERT + BERTopic": "Lebih baik -- kalimat berbeda kata tapi makna mirip bisa berdekatan"},
    {"aspek": "Jumlah cluster/topic",
     "TF-IDF + KMeans": "Ditentukan manual (k), dievaluasi lewat beberapa kandidat k",
     "SBERT + BERTopic": "Ditemukan otomatis oleh HDBSCAN, dipengaruhi min_cluster_size"},
    {"aspek": "Penanganan outlier",
     "TF-IDF + KMeans": "Tidak ada -- setiap dokumen dipaksa masuk salah satu cluster",
     "SBERT + BERTopic": "Eksplisit -- dokumen yang tidak cukup padat menjadi topic -1 (outlier)"},
    {"aspek": "Interpretabilitas",
     "TF-IDF + KMeans": "Top term per cluster (mis. dari centroid) relatif mudah dibaca",
     "SBERT + BERTopic": "Top words c-TF-IDF + representative documents, butuh pembacaan lebih dekat"},
    {"aspek": "Metrik evaluasi",
     "TF-IDF + KMeans": "silhouette, calinski_harabasz, davies_bouldin (baseline_results.csv)",
     "SBERT + BERTopic": "jumlah topik/outlier, topic_diversity, coherence (topic_model_results.csv)"},
])
display(perbandingan_metode)
print("\nCatatan: metrik kedua metode TIDAK sepenuhnya sebanding satu sama lain (berbeda ruang "
      "representasi & definisi cluster/topik) -- tabel ini untuk perbandingan KONSEPTUAL, bukan "
      "klaim satu metode 'menang' berdasarkan angka semata.")


### 13.3 Pemilihan Konfigurasi Final (HDBSCAN `min_cluster_size`)

Rubrik di bawah bersifat **alat bantu keputusan**, bukan aturan otomatis mutlak: skor
komposit menggabungkan proporsi outlier (rendah lebih baik), topic diversity (tinggi lebih
baik), dan coherence u_mass (mendekati 0 / kurang negatif lebih baik), masing-masing
dinormalisasi ke skala 0-1 lalu dirata-rata dengan bobot SAMA (1/3 masing-masing) --
bobot ini adalah **asumsi metodologis** yang bisa diubah dan didokumentasikan ulang jika
peneliti punya prioritas berbeda (mis. lebih mementingkan interpretabilitas). Konfigurasi
dengan jumlah topik reguler `< 2` otomatis didiskualifikasi (tidak ada struktur topik yang
berarti untuk dianalisis).


In [ ]:
def pilih_konfigurasi_final(df_hasil: pd.DataFrame, dataset_name: str) -> pd.Series:
    """
    Memilih satu baris (satu konfigurasi min_cluster_size) dari topic_model_results untuk
    dataset tertentu, berdasarkan skor komposit yang dijelaskan pada sel markdown di atas.
    Mengembalikan baris terpilih (pd.Series) beserta kolom skor & alasan tambahan.
    """
    subset = df_hasil[df_hasil["dataset"] == dataset_name].copy()
    subset = subset[subset["jumlah_topic"] >= 2].copy()
    if subset.empty:
        raise ValueError(f"Tidak ada konfigurasi dengan >=2 topik reguler untuk dataset {dataset_name}.")

    def normalisasi(seri, lebih_tinggi_lebih_baik=True):
        rentang = seri.max() - seri.min()
        if rentang == 0:
            return pd.Series([1.0] * len(seri), index=seri.index)
        skor = (seri - seri.min()) / rentang
        return skor if lebih_tinggi_lebih_baik else 1 - skor

    skor_outlier = normalisasi(subset["proporsi_outlier"], lebih_tinggi_lebih_baik=False)
    skor_diversity = normalisasi(subset["topic_diversity"].fillna(subset["topic_diversity"].min()),
                                  lebih_tinggi_lebih_baik=True)
    if subset["coherence"].notna().any():
        skor_coherence = normalisasi(subset["coherence"].fillna(subset["coherence"].min()),
                                      lebih_tinggi_lebih_baik=True)  # coherence mendekati 0 = lebih baik = nilai lebih besar
    else:
        skor_coherence = pd.Series([0.5] * len(subset), index=subset.index)  # netral jika coherence tak tersedia

    subset["skor_composite"] = (skor_outlier + skor_diversity + skor_coherence) / 3
    subset = subset.sort_values("skor_composite", ascending=False)
    terpilih = subset.iloc[0].copy()

    terpilih["alasan_pemilihan"] = (
        f"Skor komposit tertinggi ({terpilih['skor_composite']:.3f}) di antara konfigurasi dengan "
        f">=2 topik reguler, mempertimbangkan proporsi outlier ({terpilih['proporsi_outlier']:.1%}), "
        f"topic diversity ({terpilih['topic_diversity']}), dan coherence u_mass "
        f"({terpilih['coherence']}). Bobot ketiga komponen SAMA (1/3) -- lihat sel markdown di atas "
        f"untuk asumsi ini; peneliti dapat menimbang ulang jika prioritas berbeda."
    )
    return terpilih


final_berita = pilih_konfigurasi_final(topic_model_results, "berita")
final_ulasan = pilih_konfigurasi_final(topic_model_results, "ulasan")

final_model_selection = pd.DataFrame([final_berita, final_ulasan])[
    ["dataset", "model_id", "min_topic_size", "jumlah_topic", "proporsi_outlier",
     "topic_diversity", "coherence", "skor_composite", "alasan_pemilihan"]
]
display(final_model_selection)


In [ ]:
FINAL_MODEL_SELECTION_PATH = os.path.join(PROCESSED_DIR, "final_model_selection.csv")
final_model_selection.to_csv(FINAL_MODEL_SELECTION_PATH, index=False, encoding="utf-8-sig")
print("Tersimpan:", FINAL_MODEL_SELECTION_PATH)
print("\nPENTING: jika min_cluster_size terpilih di sini BERBEDA dari MCS_TERPILIH_BERITA/ULASAN "
      "yang dipakai di Bagian 11-12, jalankan ulang Bagian 11 (topic summary) dan Bagian 12 "
      "(analisis temporal) dengan nilai yang sesuai skor komposit ini sebelum menulis Bab IV.")


## 14. Visualisasi & Konfigurasi Eksperimen

**Tujuan:** membuat visualisasi penting (distribusi topik, outlier, tren temporal) dan mencatat
seluruh konfigurasi eksperimen ke `model_config.json` agar dapat direproduksi dan dilaporkan di
Bab III. **Input:** seluruh objek dari bagian sebelumnya. **Output:** file `.html`/`.png` di
`data/processed/figures/`, dan `data/processed/model_config.json`.

Visualisasi dibatasi pada yang benar-benar relevan (bukan seluruh kombinasi grafik yang
mungkin) sesuai batasan efisiensi. Figur interaktif BERTopic disimpan sebagai `.html` (tidak
butuh dependensi tambahan seperti kaleido untuk ekspor gambar statis).


In [ ]:
def simpan_figure_bertopic(fig, nama_file: str):
    """Menyimpan figure Plotly BERTopic sebagai HTML interaktif ke folder figures/."""
    path = os.path.join(FIGURES_DIR, nama_file)
    fig.write_html(path)
    print("Tersimpan:", path)


# --- Visualisasi bawaan BERTopic (dibungkus try/except KHUSUS untuk visualisasi opsional --
#     bukan untuk logika inti pipeline -- karena stabilitasnya bergantung environment Colab) ---
for nama_dataset, model, docs, meta_df in [
    ("berita", topic_model_berita, documents_berita, df_berita),
    ("ulasan", topic_model_ulasan, documents_ulasan, df_ulasan),
]:
    try:
        fig_topics = model.visualize_topics()
        simpan_figure_bertopic(fig_topics, f"visualize_topics_{nama_dataset}.html")
    except Exception as e:
        print(f"[{nama_dataset}] visualize_topics() dilewati: {e}")

    try:
        fig_barchart = model.visualize_barchart(top_n_topics=10)
        simpan_figure_bertopic(fig_barchart, f"visualize_barchart_{nama_dataset}.html")
    except Exception as e:
        print(f"[{nama_dataset}] visualize_barchart() dilewati: {e}")

    try:
        fig_heatmap = model.visualize_heatmap()
        simpan_figure_bertopic(fig_heatmap, f"visualize_heatmap_{nama_dataset}.html")
    except Exception as e:
        print(f"[{nama_dataset}] visualize_heatmap() dilewati: {e}")

    try:
        timestamps = pd.to_datetime(meta_df["tanggal"], errors="coerce").fillna(
            pd.to_datetime(meta_df["tahun"].astype("Int64").astype(str) + "-01-01", errors="coerce")
        )
        topics_over_time = model.topics_over_time(docs, timestamps.tolist(), nr_bins=20)
        fig_tot = model.visualize_topics_over_time(topics_over_time, top_n_topics=10)
        simpan_figure_bertopic(fig_tot, f"visualize_topics_over_time_{nama_dataset}.html")
    except Exception as e:
        print(f"[{nama_dataset}] visualize_topics_over_time() dilewati: {e}")


In [ ]:
# --- Visualisasi custom dengan matplotlib (ringan, tanpa dependensi tambahan) ---
def plot_distribusi_topic_size(topic_summary: pd.DataFrame, dataset_name: str):
    reguler = topic_summary[topic_summary["topic"] != -1].sort_values("topic_size", ascending=False)
    if reguler.empty:
        print(f"[{dataset_name}] Tidak ada topik reguler -- grafik distribusi ukuran topik dilewati.")
        return
    fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(reguler))))
    ax.barh(reguler["topic"].astype(str), reguler["topic_size"])
    ax.set_xlabel("Jumlah dokumen")
    ax.set_ylabel("Topic")
    ax.set_title(f"Distribusi ukuran topik -- {dataset_name}")
    ax.invert_yaxis()
    fig.tight_layout()
    path = os.path.join(FIGURES_DIR, f"distribusi_topic_size_{dataset_name}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)
    print("Tersimpan:", path)


def plot_proporsi_outlier(df_hasil: pd.DataFrame, dataset_name: str):
    subset = df_hasil[df_hasil["dataset"] == dataset_name]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(subset["min_topic_size"].astype(str), subset["proporsi_outlier"])
    ax.set_xlabel("min_cluster_size (HDBSCAN)")
    ax.set_ylabel("Proporsi outlier")
    ax.set_title(f"Proporsi outlier vs min_cluster_size -- {dataset_name}")
    fig.tight_layout()
    path = os.path.join(FIGURES_DIR, f"proporsi_outlier_{dataset_name}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)
    print("Tersimpan:", path)


def plot_temporal_tahun(topic_temporal: pd.DataFrame, dataset_name: str, top_n_topik: int = 5):
    tahunan = topic_temporal[(topic_temporal["granularitas"] == "tahun") & (topic_temporal["topic"] != -1)]
    if tahunan.empty:
        print(f"[{dataset_name}] Tidak ada topik reguler pada granularitas tahun -- grafik tren "
              f"temporal dilewati (semua dokumen outlier pada model yang dipakai di Bagian 11).")
        return

    topik_terbesar = (tahunan.groupby("topic")["jumlah_dokumen"].sum()
                      .nlargest(top_n_topik).index.tolist())
    fig, ax = plt.subplots(figsize=(8, 5))
    for tid in topik_terbesar:
        subset = tahunan[tahunan["topic"] == tid].sort_values("periode")
        ax.plot(subset["periode"], subset["proporsi_topic_dalam_periode"], marker="o", label=f"Topic {tid}")
    ax.set_xlabel("Tahun")
    ax.set_ylabel("Proporsi topik dalam tahun tsb.")
    ax.set_title(f"Tren proporsi topik per tahun (top {top_n_topik}) -- {dataset_name}")
    ax.legend()
    fig.tight_layout()
    path = os.path.join(FIGURES_DIR, f"tren_temporal_tahun_{dataset_name}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)
    print("Tersimpan:", path)


plot_distribusi_topic_size(topic_summary_berita, "berita")
plot_distribusi_topic_size(topic_summary_ulasan, "ulasan")
plot_proporsi_outlier(topic_model_results, "berita")
plot_proporsi_outlier(topic_model_results, "ulasan")
plot_temporal_tahun(topic_temporal_berita, "berita")
plot_temporal_tahun(topic_temporal_ulasan, "ulasan")


In [ ]:
model_config = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "random_state": RANDOM_STATE,
    "python_version": sys.version.split()[0],
    "package_versions": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "sentence_transformers": _versi_paket("sentence_transformers", "sentence-transformers"),
        "bertopic": _versi_paket("bertopic", "bertopic"),
        "hdbscan": _versi_paket("hdbscan", "hdbscan"),
        "umap_learn": _versi_paket("umap", "umap-learn"),
        "gensim": _versi_paket("gensim", "gensim") if GENSIM_AVAILABLE else None,
    },
    "device": DEVICE,
    "sbert": {
        "model_candidates": SBERT_MODEL_CANDIDATES,
        "model_used": SBERT_MODEL_NAME,
        "embedding_dim": int(EMBEDDING_DIM),
        "embedding_normalized": False,
        "similarity_metric": "cosine",
    },
    "baseline_tfidf_kmeans": {
        "k_candidates": BASELINE_K_CANDIDATES,
        "tfidf_max_features": 5000,
        "tfidf_min_df": 2,
    },
    "umap_config": UMAP_CONFIG,
    "hdbscan": {
        "min_cluster_size_candidates": HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
    },
    "model_terpilih": {
        "berita": {
            "min_cluster_size": int(final_berita["min_topic_size"]),
            "skor_composite": float(final_berita["skor_composite"]),
        },
        "ulasan": {
            "min_cluster_size": int(final_ulasan["min_topic_size"]),
            "skor_composite": float(final_ulasan["skor_composite"]),
        },
        "catatan": (
            "min_cluster_size terpilih di sini berdasarkan skor komposit Bagian 13.3. "
            "Jika Bagian 11-12 dijalankan dengan MCS_TERPILIH_* yang berbeda, perbarui "
            "MCS_TERPILIH_BERITA/ULASAN dan jalankan ulang bagian tsb agar konsisten."
        ),
    },
    "jumlah_dokumen": {"berita": len(df_berita), "ulasan": len(df_ulasan)},
}

MODEL_CONFIG_PATH = os.path.join(PROCESSED_DIR, "model_config.json")
with open(MODEL_CONFIG_PATH, "w", encoding="utf-8") as f:
    json_lib.dump(model_config, f, ensure_ascii=False, indent=2)

print("Tersimpan:", MODEL_CONFIG_PATH)
display(pd.json_normalize(model_config))


In [ ]:
print("=== DAFTAR FILE OUTPUT data/processed/ ===")
for nama_file in sorted(os.listdir(PROCESSED_DIR)):
    path_lengkap = os.path.join(PROCESSED_DIR, nama_file)
    if os.path.isfile(path_lengkap):
        ukuran_kb = os.path.getsize(path_lengkap) / 1024
        print(f"  {nama_file:<40} {ukuran_kb:>10.1f} KB")
print("\ndata/processed/figures/:")
for nama_file in sorted(os.listdir(FIGURES_DIR)):
    print(" ", nama_file)


## Ringkasan Eksperimen & Rekomendasi Konfigurasi Final

**Apa yang sudah dikerjakan:**
- Baseline TF-IDF + KMeans (5 kandidat k) sebagai pembanding metodologis, dievaluasi dengan
  3 metrik independen (`baseline_results.csv`).
- Embedding SBERT dihitung sekali per corpus dan di-cache (`embeddings_*.npy` +
  `embedding_index_*.csv`), model dipilih otomatis dari daftar kandidat yang berhasil dimuat.
- BERTopic dijalankan dengan embedding yang sudah dihitung (tidak menghitung ulang), diuji pada
  4 konfigurasi `min_cluster_size` HDBSCAN per corpus dengan UMAP yang konsisten
  (`topic_model_results.csv`).
- Evaluasi topic model tidak hanya berdasarkan jumlah topik: proporsi outlier, topic diversity,
  dan coherence (u_mass, dengan keterbatasan yang didokumentasikan) turut dipertimbangkan.
- Representasi topik (top words + representative documents) dan kolom `interpretasi_awal`
  yang sengaja dikosongkan untuk diisi peneliti (`topic_summary_*.csv`).
- Analisis temporal per tahun/bulan/tahun_bulan dengan proporsi (bukan hanya jumlah absolut)
  (`topic_temporal_*.csv`, `document_topic_*.csv`).
- Perbandingan konseptual baseline vs SBERT-BERTopic, dan perbandingan deskriptif berita vs
  ulasan (tanpa mengasumsikan nomor topik yang sama berarti topik yang sama).
- Pemilihan konfigurasi final `min_cluster_size` per corpus dengan rubrik skor komposit yang
  transparan dan dapat ditelusuri (`final_model_selection.csv`).
- Seluruh konfigurasi & versi paket dicatat ke `model_config.json` untuk Bab III.

**Rekomendasi konfigurasi final:** lihat `final_model_selection.csv` -- kolom
`alasan_pemilihan` berisi penjelasan yang bisa dikutip langsung ke Bab III/IV. Kolom ini BUKAN
klaim bahwa konfigurasi tersebut "benar secara objektif", melainkan hasil rubrik eksplisit yang
bobotnya bisa dipertimbangkan ulang oleh peneliti.

**Checklist yang perlu Anda periksa setelah menjalankan notebook ini:**
1. [ ] `topic_model_results.csv` -- apakah proporsi outlier pada konfigurasi terpilih masuk
   akal (tidak mendekati 100%)?
2. [ ] `topic_summary_berita.csv` / `topic_summary_ulasan.csv` -- baca top_words +
   representative_text_preview tiap topik reguler, lalu ISI kolom `interpretasi_awal`.
3. [ ] Bandingkan apakah `MCS_TERPILIH_BERITA` / `MCS_TERPILIH_ULASAN` pada Bagian 11 sudah
   sesuai dengan `final_model_selection.csv`; jika belum, jalankan ulang Bagian 11-14 dengan
   nilai yang benar.
4. [ ] `topic_temporal_*.csv` dan grafik di `data/processed/figures/` -- apakah tren topik dari
   waktu ke waktu masuk akal secara substantif (bukan hanya artefak jumlah data yang berubah)?
5. [ ] `baseline_results.csv` vs `topic_model_results.csv` -- gunakan `perbandingan_metode`
   (Bagian 13.2) sebagai kerangka penulisan Bab IV, bukan klaim satu metode "menang".
6. [ ] `model_config.json` -- lampirkan ke Bab III sebagai bukti reproducibility (versi
   library, random seed, konfigurasi UMAP/HDBSCAN/SBERT).
7. [ ] Periksa `data/processed/figures/*.html` di browser (visualisasi interaktif BERTopic)
   sebelum memutuskan mana yang dipakai di Bab IV -- ambil screenshot jika perlu figur statis.

**Yang TIDAK dikerjakan pada notebook ini (sengaja):**
- Dashboard Streamlit -- akan dibuat terpisah, memakai `document_topic_*.csv`,
  `topic_summary_*.csv`, dan `topic_temporal_*.csv` sebagai sumber data utamanya.
- Keputusan interpretasi topik final -- itu pekerjaan peneliti (Anda), bukan output otomatis.
